# Generate manuscript figure data (Python tier)

Runs the seven `figures/prep/figureN.py` modules with the active **Python kernel**.
Each module reads from `DATA_PATH` and writes CSVs to `SURV_PATH/results/figure_data/`. The R
rendering tier (`07_render_figures.Rmd`) consumes those CSVs.

- **Code lookups (R)**: [06a_generate_code_lookups.Rmd](06a_generate_code_lookups.Rmd) — one-time bootstrap.
- **Prep tier**: this notebook (Python).
- **Render tier**: [07_render_figures.Rmd](07_render_figures.Rmd) (R Markdown).

**Prerequisite:** `figures.prep.figure2` labels its phecode panels from the CSVs that `06a`
builds. Run `06a` once before the first `06b`, and again after a cohort rebuild. This notebook
does **not** invoke R — if those lookups are absent, figure2 falls back to raw codes and logs the
miss counts rather than failing; the preflight cell below tells you which case you are in.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path


def find_v2_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "figures").is_dir():
            return candidate
    raise RuntimeError(f"Could not find v2 root from {start}")


V2_ROOT = find_v2_root()
sys.path.insert(0, str(V2_ROOT))
from config import CODE_PATH  # noqa: E402

print(f"v2 root: {V2_ROOT}")
print(f"Python:  {sys.executable}")

# Preflight: figure2's phecode labels come from the lookups that 06a builds.
# Absent lookups are a degraded run, not a failure — say so loudly rather than silently.
_missing = [
    name for name in ("icd10_to_phecode_mapping.csv", "phecode_descriptions.csv")
    if not os.path.exists(os.path.join(CODE_PATH, name))
]
if _missing:
    print(
        "\nWARNING: missing code lookups: " + ", ".join(_missing)
        + "\n  figure2 will fall back to raw codes and disable cross-scheme event dedup."
        + "\n  Run 06a_generate_code_lookups.Rmd for manuscript-quality labels."
    )
else:
    print("Code lookups: present (06a has run).")

# Optional args for figures.prep.figure4, e.g. ["--decay", "0.1"] or ["--input", "/path/to/file.csv"]
FIGURE4_PREP_ARGS: list[str] = []


In [ ]:
PREP_MODULES = [
    ["figures.prep.figure0"],
    ["figures.prep.figure1"],
    ["figures.prep.figure2"],
    ["figures.prep.figure2_anchor"],
    ["figures.prep.figure3"],
    ["figures.prep.figure4", *FIGURE4_PREP_ARGS],
    ["figures.prep.figure5"],
]


def run_prep(args: list[str]) -> None:
    print("\n=== " + " ".join(args) + " ===", flush=True)
    subprocess.run([sys.executable, "-m", *args], cwd=V2_ROOT, check=True)


for args in PREP_MODULES:
    run_prep(args)

print("\nDone. Figure data written to $SURV_PATH/results/figure_data/.", flush=True)
print("Now run 07_render_figures.Rmd (R Markdown) to plot.", flush=True)
